# Optuna Results Viewer
PyCharmのこのセルを実行すると、Optunaのデータベースから結果を取得し、インタラクティブなテーブルとして表示します。
`target_study_id` または `target_study_name` を変更することで、動的に読み込むStudyを切り替えられます。

In [3]:
import optuna
import pandas as pd
import sqlite3

# ==========================================
# 設定: ここを書き換えて動的にStudyを指定します
# ==========================================
db_path = "optuna_qrc_nonetype.db"

# Studyの指定方法（どちらかを使用。両方Noneなら一番最新を取得します）
target_study_id = 51    # 例: 1 (データベース内のIDで指定する場合)
target_study_name = None  # 例: "qrc_lorenz_vpt..." (名前で指定する場合)

# フィルタリング設定: value（VPTなど）がこの値以上の行のみ表示します
min_value_threshold = 1.0 # 例: 5.0
# ==========================================

storage_url = f"sqlite:///{db_path}"

# 利用可能なStudy一覧を取得するヘルパー関数
def get_studies(db_file):
    try:
        with sqlite3.connect(db_file) as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT study_id, study_name FROM studies ORDER BY study_id ASC")
            return cursor.fetchall()
    except Exception as e:
        return []

studies = get_studies(db_path)

if not studies:
    print(f"データベース '{db_path}' にStudyが見つからない、またはファイルが存在しません。")
    df_view = pd.DataFrame({"Error": ["Studyが見つかりません"]})
else:
    print("【利用可能なStudy一覧】")
    for s_id, s_name in studies:
        print(f"  ID: {s_id} | Name: {s_name}")
        
    # どのStudyを読み込むか決定
    selected_name = None
    if target_study_id is not None:
        for s_id, s_name in studies:
            if s_id == target_study_id:
                selected_name = s_name
                break
        if selected_name is None:
            print(f"\n⚠️ 警告: 指定された ID '{target_study_id}' は見つかりませんでした。")
            
    elif target_study_name is not None:
        for s_id, s_name in studies:
            if s_name == target_study_name:
                selected_name = s_name
                break
        if selected_name is None:
            print(f"\n⚠️ 警告: 指定された Name '{target_study_name}' は見つかりませんでした。")

    # 指定がない、または見つからなかった場合は最新（最後）のものを選択
    if selected_name is None:
        selected_name = studies[-1][1]
        print(f"\n-> 🔄 最新のStudyを読み込みます: {selected_name}")
    else:
        print(f"\n-> ✅ 指定されたStudyを読み込みます: {selected_name}")

    # Optunaからデータを取得
    study = optuna.load_study(study_name=selected_name, storage=storage_url)
    df = study.trials_dataframe()
    
    # 見やすいようにフォーマット（COMPLETEのみ抽出し、VPTで降順ソート）
    if 'state' in df.columns:
        df = df[df['state'] == 'COMPLETE']
    if 'value' in df.columns:
        # 指定された閾値以上のものだけをフィルタリング
        if min_value_threshold is not None:
            df = df[df['value'] >= min_value_threshold]
        df = df.sort_values(by='value', ascending=False)
        
    # 不要な列を隠して見やすくする（datetime等）
    drop_cols = [c for c in df.columns if c.startswith('datetime_') or c.startswith('duration')]
    df_view = df.drop(columns=drop_cols)
    
    # 必要な列を抽出してCSVに保存
    import os
    export_cols = ['value', 'params_feature_max', 'params_feedback_scale', 'params_leak_rate']
    valid_export_cols = [c for c in export_cols if c in df_view.columns]
    if valid_export_cols:
        out_path = os.path.join("/home/yoshi/PycharmProjects/Reservoir/benchmarks", "filtered_optuna_results.csv")
        df_view[valid_export_cols].to_csv(out_path, index=False)
        print(f"\n✅ 抽出したデータを '{out_path}' に保存しました。")

# DataFrameを表示（PyCharmが綺麗なテーブルでレンダリングします）
df_view.head(50)

【利用可能なStudy一覧】
  ID: 8 | Name: qrc_vpt_minmax_nonetype_q10_Z+ZZ_poly_square
  ID: 9 | Name: qrc_vpt_minmax_nonetype_q16_Z+ZZ_poly_square
  ID: 10 | Name: qrc_vpt_minmax_nonetype_q8_Z+ZZ_poly_square
  ID: 11 | Name: qrc_vpt_minmax_nonetype_q6_Z+ZZ_poly_square
  ID: 15 | Name: qrc_vpt_crs_centered_nonetype_q10_Z+ZZ_poly_square
  ID: 16 | Name: qrc_vpt_crs_nonetype_q10_Z+ZZ_poly_square
  ID: 17 | Name: qrc_vpt_crs_centered_nonetype_q6_Z+ZZ_poly_square
  ID: 19 | Name: qrc_vpt_affine_nonetype_q6_Z+ZZ_poly_square
  ID: 20 | Name: qrc_vpt_affine_nonetype_q10_Z+ZZ_poly_square
  ID: 21 | Name: qrc_vpt_affine_nonetype_q16_Z+ZZ_poly_square
  ID: 22 | Name: qrc_vpt_minmax_nonetype_q6_Z+ZZ_poly_square_kai
  ID: 23 | Name: qrc_vpt_minmax_nonetype_q16_Z+ZZ_poly_square_kai
  ID: 24 | Name: qrc_vpt_minmax0_nonetype_q16_Z+ZZ_poly_square_kai
  ID: 25 | Name: qrc_vpt_minmax0_nonetype_q6_Z+ZZ_poly_square_kai2
  ID: 28 | Name: qrc_vpt_minmax0_nonetype_q16_Z+ZZ_poly_square_kai4
  ID: 32 | Name: qrc_vpt_minm

,number,value,params_feature_max,params_feature_min,params_feedback_scale,params_leak_rate,params_n_layers,user_attrs_best_lambda,user_attrs_correlation,user_attrs_error,...,user_attrs_status,user_attrs_truth_max,user_attrs_truth_mean,user_attrs_truth_min,user_attrs_truth_std,user_attrs_var_ratio,user_attrs_vpt_lt,user_attrs_vpt_steps,user_attrs_vpt_threshold,state
159,159,3.283133,0.533738,-0.000097,2.509779,0.182103,1,4.893901e-07,0.267338,NaN,...,success,0.532988,0.303241,0.001024,0.131416,0.993997,3.283133,545.0,0.4,COMPLETE
41,41,2.855422,0.230525,-0.000097,2.365954,0.232134,1,4.520354e-08,0.215225,NaN,...,success,0.230201,0.130948,0.000387,0.056773,0.996779,2.855422,474.0,0.4,COMPLETE
168,168,2.427711,0.509157,-0.000097,2.486505,0.177661,1,1.487352e-07,0.121254,NaN,...,success,0.508442,0.289274,0.000972,0.125365,1.001356,2.427711,403.0,0.4,COMPLETE
104,104,2.072289,0.229477,-0.000097,2.265922,0.224777,1,4.520354e-08,-0.120758,NaN,...,success,0.229155,0.130353,0.000385,0.056515,1.009288,2.072289,344.0,0.4,COMPLETE
188,188,1.873494,0.169687,-0.000097,2.577420,0.112539,1,1.487352e-07,-0.046780,NaN,...,success,0.169449,0.096378,0.000259,0.041796,0.999751,1.873494,311.0,0.4,COMPLETE
163,163,1.861446,0.562476,-0.000097,2.392635,0.133085,1,4.893901e-07,-0.004082,NaN,...,success,0.561686,0.319571,0.001084,0.138491,1.003221,1.861446,309.0,0.4,COMPLETE
226,226,1.861446,0.214415,-0.000097,2.498807,0.118295,1,1.487352e-07,0.098352,NaN,...,success,0.214114,0.121794,0.000353,0.052807,0.998840,1.861446,309.0,0.4,COMPLETE
211,211,1.861446,0.221336,-0.000097,2.504833,0.117443,1,1.487352e-07,0.096176,NaN,...,success,0.221025,0.125727,0.000368,0.054511,0.999499,1.861446,309.0,0.4,COMPLETE
257,257,1.861446,0.217356,-0.000097,2.411285,0.148104,1,4.520354e-08,0.027419,NaN,...,success,0.217051,0.123465,0.000359,0.053531,0.995323,1.861446,309.0,0.4,COMPLETE
161,161,1.771084,0.483141,-0.000097,2.430907,0.182538,1,1.487352e-07,0.111298,NaN,...,success,0.482463,0.274491,0.000918,0.118961,1.032808,1.771084,294.0,0.4,COMPLETE
